## `Real-Time Finger Counting Using CNN and ROI`

This notebook implements a real-time finger classification system using a webcam and a trained CNN model. The workflow is organized as follows:

`1. Webcam Stream and Frame Capture`

The system begins by accessing the user's webcam to continuously capture live video frames. Each frame is processed in real-time.

`2. Region of Interest (ROI) Definition`

A predefined or user-selected region on the screen (ROI) is used to focus on the area where the hand is expected to appear. This isolates the relevant image region from the full frame for further processing.

`3. ROI Preprocessing`

The ROI image is preprocessed (resized, normalized, reshaped, etc.) to match the input requirements of the CNN model. This step ensures consistent input for accurate prediction.

`4. Model Inference`

The preprocessed ROI is passed through a previously trained CNN model. The model outputs probabilities for each possible class (e.g., 0–5 fingers), and the class with the highest probability is selected as the prediction.

`5. Displaying the Result`

The predicted number of fingers is overlayed directly onto the video stream for the user to see. A rectangle may also be drawn around the ROI for visual reference.

### Import

In [2]:
from tensorflow.keras.models import load_model
import cv2
import numpy as np

## Constants

In [3]:
IMG_SIZE = 64
ROI_WIDTH = 200
ROI_HEIGHT = 200
MODEL = load_model("models/finger_count_model1.h5")

#### ROI PARAMETERS

In [4]:
roi_top_left = (80, 80)

### `mouse_callback(event, x, y, flags, param)`
  Sets the top-left corner of the ROI when the user clicks the left mouse button.

In [5]:
def mouse_callback(event, x, y, flags, param):
    """Set top-left corner of ROI on left mouse click"""
    global roi_top_left

    if event == cv2.EVENT_LBUTTONDOWN:
        roi_top_left = (x, y)
        print(f"Top-left corner selected at: {roi_top_left}")


### `get_roi(frame)`
  Extracts a fixed-size Region of Interest (ROI) starting from the selected top-left corner.

In [6]:
def get_roi(frame):
    """Extract ROI starting from top-left corner with fixed size"""
    if roi_top_left:
        x, y = roi_top_left
        x2 = x + ROI_WIDTH
        y2 = y + ROI_HEIGHT
        height, width = frame.shape[:2]
        x2 = min(x2, width)
        y2 = min(y2, height)
        return frame[y:y2, x:x2]
    return None


### `draw_roi_rectangle(frame)`
  Draws a green rectangle on the frame to visualize the selected ROI.

In [7]:
def draw_roi_rectangle(frame):
    """Draw rectangle of ROI on frame"""
    if roi_top_left:
        x, y = roi_top_left
        cv2.rectangle(frame, (x, y), (x + ROI_WIDTH, y + ROI_HEIGHT), (0, 255, 0), 2)


### `preprocess_roi(roi)`
  Preprocesses the ROI for model prediction by converting it to grayscale, blurring, thresholding, applying morphological operations, resizing, and normalizing the image.


In [8]:
def preprocess_roi(roi):
    """Preprocess ROI for model prediction"""
    gray = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(gray, (9, 9), 0)
    _, thresh = cv2.threshold(blurred, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    kernel = np.ones((5,5), np.uint8)
    morph = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, kernel, iterations=2)

    resized = cv2.resize(morph, (IMG_SIZE, IMG_SIZE))
    normalized = resized / 255.0
    reshaped = np.reshape(normalized, (1, IMG_SIZE, IMG_SIZE, 1))
    
    # To view the binarized region
    cv2.imshow("Threshold ROI", morph)

    return reshaped


### `Main loop`  
  Captures webcam frames, detects hand gestures in the selected ROI, makes predictions, and displays the result on screen.

  The windows closes when you press `q`

In [ ]:
cap = cv2.VideoCapture(0)
cv2.namedWindow("Finger Counter")
cv2.setMouseCallback("Finger Counter", mouse_callback)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame = cv2.flip(frame, 1) 

    roi = get_roi(frame)
    if roi is not None:
        input_tensor = preprocess_roi(roi)

        # Prediction
        predictions = MODEL.predict(input_tensor)
        predicted_class = np.argmax(predictions)

        # Display the result
        cv2.putText(frame, f'Fingers: {predicted_class}',
                    (roi_top_left[0], roi_top_left[1] - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

    draw_roi_rectangle(frame)

    cv2.imshow("Finger Counter", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
